# EUR/USD  ·  RL-Powered Signal Agent
### Reinforcement Learning Trading System with Live Deployment Architecture

**Why RL over rule-based signals?**
The previous vol-threshold system had a Sharpe of -0.38 on 5-minute intraday data.
Rule-based thresholds are static and can't adapt to changing market microstructure.
A PPO agent learns *when* to trade, *which direction*, and *when to stay flat* —
optimising directly for risk-adjusted return through experience.

**Pipeline:**
1. Intraday Feature Engineering (5-min aware)
2. Custom OpenAI Gym Trading Environment
3. PPO Agent Training (Proximal Policy Optimisation)
4. Backtest & Performance vs Baseline
5. Live Signal Engine — drop-in deployment class
6. Broker Integration Templates (OANDA · MT5 · Webhook)
7. Model Persistence & Retraining Schedule


## 1  ·  Setup

In [ ]:
import subprocess, sys
pkgs = ['stable-baselines3', 'gymnasium', 'shimmy', 'joblib']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], capture_output=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings, os, json, joblib
from datetime import datetime, timedelta
from collections import deque
from scipy import stats
warnings.filterwarnings('ignore')

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor

# ── Dark theme ─────────────────────────────────────────────────────────────────
C = {
    'bg':'#0D1117','panel':'#161B22','border':'#30363D','text':'#E6EDF3',
    'muted':'#8B949E','blue':'#58A6FF','green':'#3FB950','red':'#F85149',
    'orange':'#D29922','purple':'#BC8CFF','teal':'#39D353','yellow':'#E3B341',
}
plt.rcParams.update({
    'figure.facecolor':C['bg'], 'axes.facecolor':C['panel'],
    'axes.edgecolor':C['border'], 'axes.labelcolor':C['text'],
    'text.color':C['text'], 'xtick.color':C['muted'], 'ytick.color':C['muted'],
    'grid.color':C['border'], 'grid.alpha':0.5, 'font.family':'monospace',
    'axes.titlesize':12, 'figure.dpi':120, 'axes.spines.top':False,
    'axes.spines.right':False, 'legend.framealpha':0.3,
    'legend.facecolor':C['panel'], 'legend.edgecolor':C['border'],
})

print("✓ All libraries loaded")
print(f"  stable-baselines3, gymnasium, torch ready")


## 2  ·  Intraday Feature Engineering
5-minute bars require session-aware features that a daily system misses.

In [ ]:
# ── Load raw 5-min data ────────────────────────────────────────────────────────
sp = pd.read_csv("data/Eurousd.csv", parse_dates=['Date'], index_col='Date')
sp = sp.sort_index()
sp.columns = [c.strip() for c in sp.columns]

print(f"Raw bars     : {len(sp):,}")
print(f"Period       : {sp.index[0]} -> {sp.index[-1]}")
print(f"Frequency    : 5-minute intraday")

# ── Core returns ───────────────────────────────────────────────────────────────
sp['log_ret']   = np.log(sp['Close'] / sp['Close'].shift(1))
sp['ret']       = sp['Close'].pct_change()

# ── Intraday-calibrated realized vol ──────────────────────────────────────────
# 30 bars = 2.5h  |  78 bars = 6.5h (~1 session)  |  390 bars = ~2 days
sp['rv_30']    = sp['log_ret'].rolling(30).std()   * np.sqrt(26280)   # annualised
sp['rv_78']    = sp['log_ret'].rolling(78).std()   * np.sqrt(26280)
sp['rv_390']   = sp['log_ret'].rolling(390).std()  * np.sqrt(26280)

# Vol-of-vol
sp['vov']      = sp['rv_30'].rolling(30).std()

# ── Trend signals ──────────────────────────────────────────────────────────────
sp['ema_20']   = sp['Close'].ewm(span=20).mean()
sp['ema_60']   = sp['Close'].ewm(span=60).mean()
sp['ema_200']  = sp['Close'].ewm(span=200).mean()
sp['trend_s']  = np.sign(sp['ema_20'] - sp['ema_60'])       # short-term trend
sp['trend_l']  = np.sign(sp['ema_60'] - sp['ema_200'])      # long-term trend

# ── Momentum ───────────────────────────────────────────────────────────────────
sp['mom_12']   = sp['Close'].pct_change(12)   # 1h momentum
sp['mom_48']   = sp['Close'].pct_change(48)   # 4h momentum
sp['mom_390']  = sp['Close'].pct_change(390)  # ~2d momentum

# ── Session / time features (cyclically encoded) ──────────────────────────────
sp['hour']     = sp.index.hour + sp.index.minute / 60
sp['time_sin'] = np.sin(2 * np.pi * sp['hour'] / 24)
sp['time_cos'] = np.cos(2 * np.pi * sp['hour'] / 24)
sp['dow']      = sp.index.dayofweek
sp['dow_sin']  = np.sin(2 * np.pi * sp['dow'] / 5)
sp['dow_cos']  = np.cos(2 * np.pi * sp['dow'] / 5)

# ── RSI ────────────────────────────────────────────────────────────────────────
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / (loss + 1e-9)
    return 100 - 100 / (1 + rs)

sp['rsi_14'] = compute_rsi(sp['Close'], 14)
sp['rsi_norm'] = (sp['rsi_14'] - 50) / 50   # normalise to [-1, 1]

# ── Bollinger Band position ────────────────────────────────────────────────────
bb_mid   = sp['Close'].rolling(20).mean()
bb_std   = sp['Close'].rolling(20).std()
sp['bb_pos'] = (sp['Close'] - bb_mid) / (2 * bb_std + 1e-9)  # [-1,1] approx

# ── Vol percentile (rolling 390-bar window) ────────────────────────────────────
sp['vol_pct'] = sp['rv_30'].rolling(390).rank(pct=True)

# ── Z-scored return ────────────────────────────────────────────────────────────
sp['ret_z'] = (sp['log_ret'] - sp['log_ret'].rolling(78).mean()) /               (sp['log_ret'].rolling(78).std() + 1e-9)

sp.dropna(inplace=True)

FEATURE_COLS = [
    'rv_30', 'rv_78', 'rv_390', 'vov',
    'trend_s', 'trend_l',
    'mom_12', 'mom_48', 'mom_390',
    'time_sin', 'time_cos', 'dow_sin', 'dow_cos',
    'rsi_norm', 'bb_pos', 'vol_pct', 'ret_z',
]

print(f"\nClean bars   : {len(sp):,}")
print(f"Features     : {len(FEATURE_COLS)}")
print(f"Feature cols : {FEATURE_COLS}")


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 10))
fig.suptitle('Intraday Feature Overview  ·  5-min EUR/USD', fontsize=15,
             color=C['text'], fontweight='bold')

sample = sp.iloc[-2000:]   # last 2000 bars for clarity

axes[0,0].plot(sample.index, sample['rv_30'],  color=C['blue'],   lw=0.7, label='RV-30')
axes[0,0].plot(sample.index, sample['rv_78'],  color=C['orange'], lw=0.7, label='RV-78')
axes[0,0].plot(sample.index, sample['rv_390'], color=C['red'],    lw=0.7, label='RV-390')
axes[0,0].set_title('Multi-Scale Realised Volatility', color=C['text'])
axes[0,0].legend(fontsize=8)

axes[0,1].plot(sample.index, sample['Close'],   color=C['text'],   lw=0.6, label='Price')
axes[0,1].plot(sample.index, sample['ema_20'],  color=C['blue'],   lw=0.8, label='EMA-20')
axes[0,1].plot(sample.index, sample['ema_60'],  color=C['orange'], lw=0.8, label='EMA-60')
axes[0,1].plot(sample.index, sample['ema_200'], color=C['red'],    lw=0.8, label='EMA-200', alpha=0.7)
axes[0,1].set_title('Price & EMA Trend Filters', color=C['text'])
axes[0,1].legend(fontsize=8)

axes[1,0].plot(sample.index, sample['rsi_14'], color=C['purple'], lw=0.7)
axes[1,0].axhline(70, color=C['red'],   lw=0.8, ls='--', alpha=0.7)
axes[1,0].axhline(30, color=C['green'], lw=0.8, ls='--', alpha=0.7)
axes[1,0].axhline(50, color=C['muted'], lw=0.5, ls=':')
axes[1,0].set_title('RSI (14)', color=C['text']); axes[1,0].set_ylim(0, 100)

axes[1,1].plot(sample.index, sample['bb_pos'], color=C['teal'], lw=0.7)
axes[1,1].axhline( 1, color=C['red'],   lw=0.8, ls='--', alpha=0.7)
axes[1,1].axhline(-1, color=C['green'], lw=0.8, ls='--', alpha=0.7)
axes[1,1].axhline( 0, color=C['muted'], lw=0.5, ls=':')
axes[1,1].set_title('Bollinger Band Position', color=C['text'])

axes[2,0].plot(sample.index, sample['vol_pct'], color=C['yellow'], lw=0.7)
axes[2,0].axhline(0.7, color=C['red'],   lw=1, ls='--', label='70th pct')
axes[2,0].axhline(0.3, color=C['green'], lw=1, ls='--', label='30th pct')
axes[2,0].fill_between(sample.index, 0.7, 1.0,  alpha=0.08, color=C['red'])
axes[2,0].fill_between(sample.index, 0.0, 0.3,  alpha=0.08, color=C['green'])
axes[2,0].set_title('Vol Percentile Rank (390-bar)', color=C['text'])
axes[2,0].legend(fontsize=8)

axes[2,1].plot(sample.index, sample['ret_z'].clip(-4, 4), color=C['blue'], lw=0.4, alpha=0.8)
axes[2,1].axhline(0, color=C['muted'], lw=0.5)
axes[2,1].set_title('Z-Scored Returns (clipped ±4)', color=C['text'])

for ax in axes.flatten():
    ax.tick_params(colors=C['muted'])

plt.tight_layout()
plt.savefig('feature_overview.png', dpi=130, bbox_inches='tight', facecolor=C['bg'])
plt.show()


## 3  ·  Custom Trading Environment

The agent interacts with a Gymnasium environment at each 5-min bar:
- **State**: 17 normalised features + current position + unrealised PnL
- **Action**: `{0 = Flat, 1 = Long, 2 = Short}`
- **Reward**: step PnL − transaction cost − holding penalty in low-vol regime


In [ ]:
class EURUSDTradingEnv(gym.Env):
    """
    5-minute EUR/USD RL Trading Environment.

    Observation  : [17 features | position | unrealised_pnl_norm]
    Action       : Discrete(3) — 0=Flat  1=Long  2=Short
    Reward       : net_step_pnl − cost − regime_penalty
    """
    metadata = {'render_modes': ['human']}

    ACTION_MAP = {0: 0, 1: 1, 2: -1}   # action -> position direction

    def __init__(self, data: pd.DataFrame, feature_cols: list,
                 transaction_cost: float = 0.00005,  # 0.5 pip
                 window: int = 1,
                 max_holding: int = 48,              # 4h max hold
                 reward_scaling: float = 1000.0):

        super().__init__()
        self.data             = data.reset_index(drop=True)
        self.feature_cols     = feature_cols
        self.tc               = transaction_cost
        self.window           = window
        self.max_holding      = max_holding
        self.reward_scaling   = reward_scaling
        self.n_features       = len(feature_cols) + 2   # +position +upnl

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.n_features,), dtype=np.float32)
        self.action_space = spaces.Discrete(3)

        self._cursor   = 0
        self._position = 0      # -1, 0, 1
        self._entry_px = 0.0
        self._hold_bars= 0
        self._equity   = 1.0
        self._peak     = 1.0

    # ── Gym API ──────────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._cursor    = self.window
        self._position  = 0
        self._entry_px  = 0.0
        self._hold_bars = 0
        self._equity    = 1.0
        self._peak      = 1.0
        return self._get_obs(), {}

    def step(self, action):
        direction = self.ACTION_MAP[action]
        row       = self.data.iloc[self._cursor]
        price     = float(row['Close'])
        ret       = float(row['ret']) if not np.isnan(row['ret']) else 0.0

        # Transaction cost on position change
        cost = self.tc if direction != self._position else 0.0

        # Step PnL
        step_pnl = self._position * ret - cost

        # Reward shaping
        vol_regime_penalty = 0.0
        if row['vol_pct'] < 0.25 and self._position != 0:
            vol_regime_penalty = 0.0001   # discourage trading in dead markets

        max_hold_penalty = 0.0
        if self._hold_bars > self.max_holding and self._position != 0:
            max_hold_penalty = 0.0002     # encourage position rotation

        reward = (step_pnl - vol_regime_penalty - max_hold_penalty) * self.reward_scaling

        # Update state
        if direction != self._position:
            self._position  = direction
            self._entry_px  = price
            self._hold_bars = 0
        else:
            self._hold_bars += 1

        self._equity *= (1 + step_pnl)
        self._peak    = max(self._peak, self._equity)

        self._cursor += 1
        done = self._cursor >= len(self.data) - 1

        return self._get_obs(), float(reward), done, False, {
            'pnl': step_pnl, 'position': self._position, 'equity': self._equity,
        }

    # ── Helpers ──────────────────────────────────────────────────────────────
    def _get_obs(self):
        row  = self.data.iloc[self._cursor]
        feat = row[self.feature_cols].values.astype(np.float32)
        feat = np.nan_to_num(feat, nan=0.0, posinf=1.0, neginf=-1.0)

        upnl = 0.0
        if self._position != 0 and self._entry_px > 0:
            upnl = self._position * (float(row['Close']) - self._entry_px) / self._entry_px

        extra = np.array([float(self._position), upnl], dtype=np.float32)
        return np.concatenate([feat, extra])

    def render(self):
        pass

print("✓ EURUSDTradingEnv defined")
print(f"  Observation : {len(FEATURE_COLS)+2} dims")
print(f"  Actions     : {{0: Flat, 1: Long, 2: Short}}")
print(f"  Reward      : step PnL - vol_regime_penalty - max_hold_penalty")


## 4  ·  Data Split & Environment Setup

In [ ]:
n      = len(sp)
t_end  = int(n * 0.65)
v_end  = int(n * 0.80)

train_df = sp.iloc[:t_end].copy().reset_index(drop=True)
val_df   = sp.iloc[t_end:v_end].copy().reset_index(drop=True)
test_df  = sp.iloc[v_end:].copy().reset_index(drop=True)

print(f"Train : {len(train_df):,} bars  ({len(train_df)/12:.0f}h  /  {len(train_df)/2340:.1f} trading months)")
print(f"Val   : {len(val_df):,} bars  ({len(val_df)/12:.0f}h)")
print(f"Test  : {len(test_df):,} bars  ({len(test_df)/12:.0f}h)")

def make_env(df, seed=0):
    def _init():
        env = EURUSDTradingEnv(df, FEATURE_COLS)
        env = Monitor(env)
        return env
    return _init

train_env = DummyVecEnv([make_env(train_df)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.)

eval_env  = DummyVecEnv([make_env(val_df)])
eval_env  = VecNormalize(eval_env,  norm_obs=True, norm_reward=False, clip_obs=10.,
                          training=False)

print("\n✓ Environments created with VecNormalize (obs normalisation)")


## 5  ·  PPO Agent
Proximal Policy Optimisation — stable, sample-efficient, works well for finance.
Network: MLP `[256, 256]` for both policy and value heads.


In [ ]:
class RewardLoggerCallback(BaseCallback):
    """Logs episode rewards during training for plotting."""
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
                self.episode_lengths.append(info['episode']['l'])
        return True

policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
)

model = PPO(
    policy          = 'MlpPolicy',
    env             = train_env,
    learning_rate   = 3e-4,
    n_steps         = 1024,
    batch_size      = 128,
    n_epochs        = 10,
    gamma           = 0.99,
    gae_lambda      = 0.95,
    clip_range      = 0.2,
    ent_coef        = 0.005,      # entropy bonus encourages exploration
    vf_coef         = 0.5,
    max_grad_norm   = 0.5,
    policy_kwargs   = policy_kwargs,
    verbose         = 0,
    seed            = 42,
)

reward_cb  = RewardLoggerCallback()

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path = './models/',
    log_path             = './models/logs/',
    eval_freq            = 5000,
    n_eval_episodes      = 3,
    deterministic        = True,
    verbose              = 0,
)

print("✓ PPO agent initialised")
print(f"  Policy network : MLP [256, 256]")
print(f"  Value network  : MLP [256, 256]")
print(f"  Entropy coef   : 0.005  (exploration incentive)")
print(f"  Clip range     : 0.20")
print(f"  Batch size     : 128")


## 6  ·  Training
Typically 3-5 minutes on CPU for 100k timesteps.

In [ ]:
import time
os.makedirs('models', exist_ok=True)

TOTAL_TIMESTEPS = 100_000

print(f"Training PPO for {TOTAL_TIMESTEPS:,} timesteps ...")
print(f"Expected time: ~3-5 minutes on CPU")
print()

t0 = time.time()
model.learn(
    total_timesteps = TOTAL_TIMESTEPS,
    callback        = [reward_cb, eval_cb],
    progress_bar    = True,
)
elapsed = time.time() - t0

# Save final model + normalisation stats
model.save('models/ppo_eurusd_final')
train_env.save('models/vecnorm_stats.pkl')

print(f"\n✓ Training complete in {elapsed:.0f}s")
print(f"  Episodes logged : {len(reward_cb.episode_rewards)}")
if reward_cb.episode_rewards:
    print(f"  Avg reward      : {np.mean(reward_cb.episode_rewards[-20:]):.4f}  (last 20 ep)")


In [ ]:
if reward_cb.episode_rewards:
    rewards = np.array(reward_cb.episode_rewards)
    window  = max(1, len(rewards)//20)
    smooth  = pd.Series(rewards).rolling(window, min_periods=1).mean()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('PPO Training Curve', fontsize=14, color=C['text'], fontweight='bold')

    axes[0].plot(rewards, color=C['blue'], alpha=0.3, linewidth=0.5, label='Episode reward')
    axes[0].plot(smooth,  color=C['green'], linewidth=1.5, label=f'Smoothed ({window}-ep)')
    axes[0].axhline(0, color=C['muted'], linewidth=0.6, linestyle='--')
    axes[0].set_title('Episode Rewards', color=C['text'])
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Total Reward')
    axes[0].legend(fontsize=9)

    lengths = np.array(reward_cb.episode_lengths)
    axes[1].plot(lengths, color=C['purple'], alpha=0.4, linewidth=0.5)
    axes[1].plot(pd.Series(lengths).rolling(window, min_periods=1).mean(),
                 color=C['orange'], linewidth=1.5)
    axes[1].set_title('Episode Lengths (bars)', color=C['text'])
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Bars')

    for ax in axes:
        ax.tick_params(colors=C['muted'])

    plt.tight_layout()
    plt.savefig('training_curve.png', dpi=130, bbox_inches='tight', facecolor=C['bg'])
    plt.show()
else:
    print("No episode data (environment may not have completed an episode during training)")


## 7  ·  Backtest on Held-Out Test Set

In [ ]:
def backtest_agent(model, data, feature_cols, tc=0.00005, deterministic=True):
    """Run trained agent on data, return results DataFrame."""
    env     = EURUSDTradingEnv(data, feature_cols)
    obs, _  = env.reset()

    records = []
    done    = False

    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, done, _, info = env.step(int(action))
        records.append({
            'action':    int(action),
            'position':  info['position'],
            'pnl':       info['pnl'],
            'equity':    info['equity'],
        })

    df_res = data.iloc[1:len(records)+1].copy()
    for k in ['action','position','pnl','equity']:
        df_res[k] = [r[k] for r in records]

    df_res['bh_equity'] = (1 + df_res['ret'].fillna(0)).cumprod()
    df_res['peak']      = df_res['equity'].cummax()
    df_res['drawdown']  = df_res['equity'] / df_res['peak'] - 1

    return df_res

# Load best model (saved by EvalCallback)
best_path = 'models/best_model' if os.path.exists('models/best_model.zip') else 'models/ppo_eurusd_final'
best_model = PPO.load(best_path)
print(f"Loaded model: {best_path}")

# Backtest on test set
test_res = backtest_agent(best_model, test_df, FEATURE_COLS)

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(df, label='Agent'):
    ret  = df['pnl']
    eq   = df['equity']
    dd   = df['drawdown']
    bhr  = df['ret'].fillna(0)
    ann  = 26280   # 5-min bars per year

    ann_ret  = eq.iloc[-1] ** (ann/len(eq)) - 1 if len(eq)>1 else 0
    ann_vol  = ret.std() * np.sqrt(ann)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else 0
    neg      = ret[ret < 0]
    sortino  = ann_ret / (neg.std()*np.sqrt(ann)) if len(neg)>1 else 0
    max_dd   = dd.min()
    calmar   = ann_ret / abs(max_dd) if max_dd != 0 else 0
    win_rate = (ret > 0).mean()
    pf_num   = ret[ret>0].sum(); pf_den = abs(ret[ret<0].sum())
    pf       = pf_num/pf_den if pf_den>0 else np.inf
    n_trades = (df['position'].diff().abs() > 0).sum()
    var95    = np.percentile(ret, 5)
    cvar95   = ret[ret<=var95].mean()

    bh_ann   = (df['bh_equity'].iloc[-1])**(ann/len(df)) - 1
    bh_vol   = bhr.std() * np.sqrt(ann)
    bh_sharpe= bh_ann/bh_vol if bh_vol>0 else 0

    print(f"{'='*55}")
    print(f"  {label} PERFORMANCE  (Test Set)")
    print(f"{'='*55}")
    print(f"  Annual Return     {ann_ret:>10.2%}   |  B&H: {bh_ann:.2%}")
    print(f"  Annual Vol        {ann_vol:>10.2%}   |  B&H: {bh_vol:.2%}")
    print(f"  Sharpe Ratio      {sharpe:>10.3f}   |  B&H: {bh_sharpe:.3f}")
    print(f"  Sortino Ratio     {sortino:>10.3f}")
    print(f"  Calmar Ratio      {calmar:>10.3f}")
    print(f"  Max Drawdown      {max_dd:>10.2%}")
    print(f"  Win Rate          {win_rate:>10.2%}")
    print(f"  Profit Factor     {pf:>10.2f}")
    print(f"  VaR (95%)         {var95:>10.4f}")
    print(f"  CVaR (95%)        {cvar95:>10.4f}")
    print(f"  Total Trades      {n_trades:>10,}")
    print(f"{'='*55}")
    return dict(ann_ret=ann_ret, sharpe=sharpe, sortino=sortino,
                calmar=calmar, max_dd=max_dd, win_rate=win_rate,
                pf=pf, n_trades=n_trades)

test_metrics = compute_metrics(test_res, 'RL AGENT')


In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('RL Agent  ·  Test Set Performance Dashboard', fontsize=16,
             color=C['text'], fontweight='bold')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Equity curve
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(test_res.index, test_res['equity'],    color=C['green'], lw=1.5, label='RL Agent', zorder=3)
ax1.plot(test_res.index, test_res['bh_equity'], color=C['blue'],  lw=1.0, label='Buy & Hold', alpha=0.7)
ax1.fill_between(test_res.index, 1, test_res['equity'],
                 where=test_res['equity']>=1, alpha=0.12, color=C['green'])
ax1.fill_between(test_res.index, 1, test_res['equity'],
                 where=test_res['equity']<1,  alpha=0.12, color=C['red'])
ax1.axhline(1, color=C['muted'], lw=0.7, ls='--')
ax1.set_title('Equity Curve  ·  RL Agent vs Buy & Hold', color=C['text'])
ax1.set_ylabel('Portfolio Value'); ax1.legend(fontsize=10)

# 2. Drawdown
ax2 = fig.add_subplot(gs[1, :2])
dd = pd.to_numeric(test_res['drawdown'], errors='coerce').fillna(0).values
ax2.fill_between(test_res.index, dd*100, 0, alpha=0.6, color=C['red'])
ax2.plot(test_res.index, dd*100, color=C['red'], lw=0.6)
ax2.set_title('Drawdown (%)', color=C['text']); ax2.set_ylabel('DD %')

# 3. Actions (sampled)
ax3 = fig.add_subplot(gs[1, 2])
action_counts = test_res['action'].value_counts().sort_index()
labels = {0:'Flat', 1:'Long', 2:'Short'}
colors_a= [C['muted'], C['green'], C['red']]
ax3.bar([labels[i] for i in action_counts.index],
        action_counts.values,
        color=[colors_a[i] for i in action_counts.index], alpha=0.85, edgecolor=C['border'])
ax3.set_title('Action Distribution', color=C['text'])

# 4. Rolling Sharpe
ax4 = fig.add_subplot(gs[2, 0])
ann_f = 26280
roll_sh = (test_res['pnl'].rolling(390).mean() /
           (test_res['pnl'].rolling(390).std() + 1e-9)) * np.sqrt(ann_f)
ax4.plot(test_res.index, roll_sh, color=C['purple'], lw=0.8)
ax4.axhline(0, color=C['muted'], lw=0.7, ls='--')
ax4.axhline(1, color=C['green'], lw=0.7, ls=':', alpha=0.7)
ax4.fill_between(test_res.index, roll_sh, 0,
                 where=roll_sh>=0, alpha=0.15, color=C['green'])
ax4.fill_between(test_res.index, roll_sh, 0,
                 where=roll_sh<0,  alpha=0.15, color=C['red'])
ax4.set_title('Rolling Sharpe (390-bar)', color=C['text'])

# 5. Return distribution
ax5 = fig.add_subplot(gs[2, 1])
ax5.hist(test_res['pnl'],        bins=100, density=True, color=C['green'], alpha=0.6, label='RL Agent')
ax5.hist(test_res['ret'].fillna(0), bins=100, density=True, color=C['blue'],  alpha=0.35, label='B&H')
v95 = np.percentile(test_res['pnl'], 5)
ax5.axvline(v95, color=C['red'], lw=1.5, ls='--', label=f'VaR95: {v95:.5f}')
ax5.set_title('Return Distribution', color=C['text']); ax5.legend(fontsize=8)

# 6. Position over time (sample)
ax6 = fig.add_subplot(gs[2, 2])
tail = test_res.iloc[-500:]
ax6.step(tail.index, tail['position'], color=C['yellow'], lw=0.8, where='post')
ax6.fill_between(tail.index, tail['position'], 0,
                 where=tail['position']>0, alpha=0.3, color=C['green'], step='post')
ax6.fill_between(tail.index, tail['position'], 0,
                 where=tail['position']<0, alpha=0.3, color=C['red'],   step='post')
ax6.axhline(0, color=C['muted'], lw=0.5)
ax6.set_title('Agent Position (last 500 bars)', color=C['text'])
ax6.set_yticks([-1,0,1]); ax6.set_yticklabels(['Short','Flat','Long'])

for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
    ax.tick_params(colors=C['muted'])

plt.savefig('rl_performance.png', dpi=130, bbox_inches='tight', facecolor=C['bg'])
plt.show()


## 8  ·  Live Signal Engine
Drop-in deployment class. Feed it one bar at a time from any data source
and it returns a structured signal ready for order execution.


In [ ]:
class LiveSignalEngine:
    """
    Production-ready live signal engine for EUR/USD.

    Usage:
        engine = LiveSignalEngine('models/best_model', 'models/vecnorm_stats.pkl')

        # On each new 5-min bar from your data feed:
        signal = engine.update({
            'timestamp': '2026-04-10 10:05:00',
            'Open': 1.0820, 'High': 1.0825,
            'Low': 1.0818, 'Close': 1.0823,
        })
        print(signal)
        # -> {'signal': 'LONG', 'position': 1, 'confidence': 0.87,
        #     'timestamp': '...', 'bar_count': 412}
    """

    SIGNAL_MAP = {0: 'FLAT', 1: 'LONG', 2: 'SHORT'}
    WARM_UP    = 400   # bars needed before first signal (longest feature window)

    def __init__(self, model_path: str, vecnorm_path: str = None):
        self.model     = PPO.load(model_path)
        self.buffer    = deque(maxlen=500)
        self._position = 0
        self._entry_px = None
        self._bar_count= 0
        print(f"LiveSignalEngine ready | model: {model_path}")

    def update(self, bar: dict) -> dict:
        """
        Feed one OHLCV bar dict. Returns signal dict when warmed up,
        else returns None with a warm-up notice.
        """
        self.buffer.append(bar)
        self._bar_count += 1

        if self._bar_count < self.WARM_UP:
            return {'signal': 'WARMING_UP',
                    'bars_remaining': self.WARM_UP - self._bar_count}

        obs   = self._build_observation()
        if obs is None:
            return {'signal': 'ERROR', 'reason': 'feature computation failed'}

        action, state = self.model.predict(obs, deterministic=True)
        action_probs  = self._get_action_probs(obs)

        signal_name   = self.SIGNAL_MAP[int(action)]
        confidence    = float(action_probs[int(action)])

        # Track position
        prev_position = self._position
        self._position = {0: 0, 1: 1, 2: -1}[int(action)]

        close = float(bar['Close'])
        unrealised_pnl = 0.0
        if self._entry_px and self._position != 0:
            unrealised_pnl = self._position * (close - self._entry_px) / self._entry_px

        if self._position != prev_position:
            self._entry_px = close if self._position != 0 else None

        return {
            'signal':          signal_name,
            'position':        self._position,
            'confidence':      round(confidence, 4),
            'unrealised_pnl':  round(unrealised_pnl, 6),
            'position_changed':self._position != prev_position,
            'timestamp':       bar.get('timestamp', ''),
            'close':           close,
            'bar_count':       self._bar_count,
        }

    def _build_observation(self):
        """Compute features from buffer and return observation array."""
        try:
            df = pd.DataFrame(list(self.buffer))
            df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
            df['log_ret']  = np.log(df['Close'] / df['Close'].shift(1))
            df['ret']      = df['Close'].pct_change()
            df['rv_30']    = df['log_ret'].rolling(30).std()  * np.sqrt(26280)
            df['rv_78']    = df['log_ret'].rolling(78).std()  * np.sqrt(26280)
            df['rv_390']   = df['log_ret'].rolling(390).std() * np.sqrt(26280)
            df['vov']      = df['rv_30'].rolling(30).std()
            df['ema_20']   = df['Close'].ewm(span=20).mean()
            df['ema_60']   = df['Close'].ewm(span=60).mean()
            df['ema_200']  = df['Close'].ewm(span=200).mean()
            df['trend_s']  = np.sign(df['ema_20'] - df['ema_60'])
            df['trend_l']  = np.sign(df['ema_60'] - df['ema_200'])
            df['mom_12']   = df['Close'].pct_change(12)
            df['mom_48']   = df['Close'].pct_change(48)
            df['mom_390']  = df['Close'].pct_change(390)

            try:
                hour = pd.to_datetime(df.get('timestamp', pd.RangeIndex(len(df)))).dt.hour
            except:
                hour = pd.Series(np.zeros(len(df)))

            df['time_sin'] = np.sin(2 * np.pi * hour / 24)
            df['time_cos'] = np.cos(2 * np.pi * hour / 24)
            df['dow_sin']  = 0.0; df['dow_cos'] = 1.0   # fallback

            delta = df['Close'].diff()
            gain  = delta.clip(lower=0).rolling(14).mean()
            loss  = (-delta.clip(upper=0)).rolling(14).mean()
            df['rsi_norm'] = ((100 - 100/(1 + gain/(loss+1e-9))) - 50) / 50

            bb_mid = df['Close'].rolling(20).mean()
            bb_std = df['Close'].rolling(20).std()
            df['bb_pos']   = (df['Close'] - bb_mid) / (2*bb_std + 1e-9)
            df['vol_pct']  = df['rv_30'].rolling(390).rank(pct=True)
            df['ret_z']    = (df['log_ret'] - df['log_ret'].rolling(78).mean()) /                               (df['log_ret'].rolling(78).std() + 1e-9)

            last     = df.iloc[-1]
            features = last[FEATURE_COLS].values.astype(np.float32)
            features = np.nan_to_num(features, nan=0.0, posinf=1.0, neginf=-1.0)

            upnl = 0.0
            if self._entry_px and self._position != 0:
                upnl = self._position * (float(last['Close']) - self._entry_px) / self._entry_px

            obs = np.concatenate([features, [float(self._position), upnl]])
            return obs.reshape(1, -1)

        except Exception as e:
            print(f"Feature error: {e}")
            return None

    def _get_action_probs(self, obs):
        """Softmax over policy logits — proxy for confidence."""
        try:
            import torch
            with torch.no_grad():
                obs_t    = torch.FloatTensor(obs)
                logits   = self.model.policy.action_net(
                    self.model.policy.mlp_extractor(
                        self.model.policy.features_extractor(obs_t)
                    )[0]
                )
                probs = torch.softmax(logits, dim=-1).numpy()[0]
            return probs
        except:
            return np.ones(3) / 3

    def get_status(self) -> dict:
        """Current engine status — useful for monitoring dashboards."""
        return {
            'position':   self.SIGNAL_MAP.get(self._position+1 if self._position==-1 else self._position, 'FLAT'),
            'bar_count':  self._bar_count,
            'buffer_len': len(self.buffer),
            'warmed_up':  self._bar_count >= self.WARM_UP,
        }

print("✓ LiveSignalEngine class defined")
print()
print("  Instantiate:")
print("    engine = LiveSignalEngine('models/best_model')")
print()
print("  On each bar:")
print("    signal = engine.update({'timestamp': ..., 'Close': 1.0823, ...})")


## 9  ·  Broker Integration Templates

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TEMPLATE A: OANDA REST API
# ══════════════════════════════════════════════════════════════════════════════
OANDA_TEMPLATE = '''
import oandapyV20
import oandapyV20.endpoints.orders as orders
import oandapyV20.endpoints.instruments as instruments
from live_signal_engine import LiveSignalEngine

API_KEY     = "your-oanda-api-key"
ACCOUNT_ID  = "your-account-id"
INSTRUMENT  = "EUR_USD"
UNITS       = 10000   # position size

client = oandapyV20.API(access_token=API_KEY, environment="practice")
engine = LiveSignalEngine("models/best_model")

def fetch_latest_bar():
    r   = instruments.InstrumentsCandles(INSTRUMENT,
            params={"count":1,"granularity":"M5"})
    rv  = client.request(r)
    c   = rv["candles"][-1]
    return {"timestamp": c["time"], "Open": float(c["mid"]["o"]),
            "High": float(c["mid"]["h"]), "Low": float(c["mid"]["l"]),
            "Close": float(c["mid"]["c"])}

def execute_signal(signal: dict):
    direction = signal["signal"]
    if not signal["position_changed"] or direction == "WARMING_UP":
        return
    units = UNITS if direction == "LONG" else (-UNITS if direction == "SHORT" else 0)
    if units == 0:
        # Send market close order
        pass
    else:
        body = {"order": {"units": str(units), "instrument": INSTRUMENT,
                          "timeInForce": "FOK", "type": "MARKET"}}
        r = orders.OrderCreate(ACCOUNT_ID, data=body)
        client.request(r)
        print(f"Order sent: {direction}  {units} units")

# Run on schedule (every 5 min)
import time, schedule
def run():
    bar    = fetch_latest_bar()
    signal = engine.update(bar)
    print(signal)
    execute_signal(signal)

schedule.every(5).minutes.do(run)
while True:
    schedule.run_pending()
    time.sleep(1)
'''

# ══════════════════════════════════════════════════════════════════════════════
# TEMPLATE B: MetaTrader 5 (MT5)
# ══════════════════════════════════════════════════════════════════════════════
MT5_TEMPLATE = '''
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
from live_signal_engine import LiveSignalEngine

mt5.initialize()
engine = LiveSignalEngine("models/best_model")
SYMBOL = "EURUSD"; LOT = 0.1

def fetch_bar():
    rates = mt5.copy_rates_from_pos(SYMBOL, mt5.TIMEFRAME_M5, 0, 1)
    r     = rates[0]
    return {"timestamp": str(datetime.fromtimestamp(r["time"])),
            "Open": r["open"], "High": r["high"],
            "Low": r["low"],  "Close": r["close"]}

def execute(signal):
    if not signal.get("position_changed"): return
    direction = signal["signal"]
    mt5.positions_get(symbol=SYMBOL)  # close existing
    if direction == "LONG":
        mt5.order_send({"action": mt5.TRADE_ACTION_DEAL, "symbol": SYMBOL,
                        "volume": LOT, "type": mt5.ORDER_TYPE_BUY,
                        "price": mt5.symbol_info_tick(SYMBOL).ask})
    elif direction == "SHORT":
        mt5.order_send({"action": mt5.TRADE_ACTION_DEAL, "symbol": SYMBOL,
                        "volume": LOT, "type": mt5.ORDER_TYPE_SELL,
                        "price": mt5.symbol_info_tick(SYMBOL).bid})

import time
while True:
    bar    = fetch_bar()
    signal = engine.update(bar)
    print(signal)
    execute(signal)
    time.sleep(300)   # 5 minutes
'''

# ══════════════════════════════════════════════════════════════════════════════
# TEMPLATE C: REST Webhook (FastAPI) — push signals to any downstream system
# ══════════════════════════════════════════════════════════════════════════════
WEBHOOK_TEMPLATE = '''
from fastapi import FastAPI
from pydantic import BaseModel
from live_signal_engine import LiveSignalEngine

app    = FastAPI(title="EUR/USD Signal API")
engine = LiveSignalEngine("models/best_model")

class Bar(BaseModel):
    timestamp: str
    Open: float; High: float; Low: float; Close: float

@app.post("/signal")
def get_signal(bar: Bar):
    return engine.update(bar.dict())

@app.get("/status")
def status():
    return engine.get_status()

# Run: uvicorn signal_api:app --reload --port 8000
# POST to http://localhost:8000/signal with OHLCV JSON bar
'''

print("=" * 60)
print("  BROKER INTEGRATION TEMPLATES")
print("=" * 60)
print("\n  A)  OANDA REST API   → copy OANDA_TEMPLATE")
print("  B)  MetaTrader 5     → copy MT5_TEMPLATE")
print("  C)  Webhook/FastAPI  → copy WEBHOOK_TEMPLATE")
print()
print("  All templates use the same LiveSignalEngine class.")
print("  Swap broker by changing only the fetch_bar() and execute() functions.")
print()
print("  Install per template:")
print("    OANDA : pip install oandapyV20 schedule")
print("    MT5   : pip install MetaTrader5")
print("    API   : pip install fastapi uvicorn")


## 10  ·  Model Persistence & Retraining

In [ ]:
import json
from datetime import datetime

# Save model artefacts
os.makedirs('models', exist_ok=True)
model.save('models/ppo_eurusd_final')
train_env.save('models/vecnorm_stats.pkl')

# Save feature config (so retraining uses same features)
config = {
    'feature_cols':       FEATURE_COLS,
    'n_features':         len(FEATURE_COLS) + 2,
    'transaction_cost':   0.00005,
    'max_holding_bars':   48,
    'total_timesteps':    TOTAL_TIMESTEPS,
    'trained_on':         datetime.now().isoformat(),
    'train_bars':         len(train_df),
    'test_sharpe':        round(test_metrics['sharpe'], 4),
    'test_ann_return':    round(test_metrics['ann_ret'], 4),
}

with open('models/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("=" * 55)
print("  SAVED ARTEFACTS")
print("=" * 55)
print("  models/ppo_eurusd_final.zip  - PPO weights")
print("  models/vecnorm_stats.pkl     - obs normalisation")
print("  models/model_config.json     - feature/training config")
print("  models/best_model.zip        - best val checkpoint")
print()
print("  RETRAINING SCHEDULE (recommended)")
print("=" * 55)
print("  Weekly  : retrain on rolling 3-month window")
print("  Trigger : if 30d rolling Sharpe < 0 on live signals")
print("  Method  : model.learn(..., reset_num_timesteps=False)")
print("            to continue from current weights (warm start)")
print()
print("  LOAD FOR DEPLOYMENT")
print("=" * 55)
print("  from stable_baselines3 import PPO")
print("  model = PPO.load('models/best_model')")
print("  engine = LiveSignalEngine('models/best_model')")
print()

with open('models/model_config.json') as f:
    cfg = json.load(f)
print("  Config written:")
for k, v in cfg.items():
    print(f"    {k:<22} {v}")
